In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task
Process the paper URLs from the "SB_publication_PMC.csv" file ["SB_publication_PMC.csv"], download the full text XML for each paper using the provided API, extract specific metadata (title, authors, year, journal, abstract, keywords, mission) from the XML, organize the extracted data and original XML into a specified folder structure (`./public/data/papers/{pmc_id}/`), and create a master JSON file (`./public/data/publications.json`) containing all the extracted metadata.

## Load the data

### Subtask:
Load the `SB_publication_PMC.csv` file into a pandas DataFrame.


**Reasoning**:
The first step is to load the data from the CSV file into a pandas DataFrame for further processing. This involves importing the pandas library, reading the CSV file, and displaying the head of the DataFrame to verify the loading process.



In [ ]:
import pandas as pd

df = pd.read_csv('/content/SB_publication_PMC.csv')
display(df.head())

## Extract paper ids

### Subtask:
Extract the paper ID from each URL in the DataFrame.


**Reasoning**:
Define a function to extract PMC IDs from URLs and apply it to the DataFrame.



In [ ]:
def extract_pmc_id(url):
    """Extracts the PMC ID from a given URL."""
    if isinstance(url, str):
        parts = url.split('/PMC')
        if len(parts) > 1:
            pmc_part = parts[-1]
            # Assuming the PMC ID is immediately after '/PMC' and before any other path elements
            pmc_id = pmc_part.split('/')[0]
            return pmc_id
    return None

df['PMC_ID'] = df['Link'].apply(extract_pmc_id)
display(df.head())

,Title,Link,PMC_ID
0,Mice in Bion-M 1 space mission: training and s...,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4...,4136787
1,Microgravity induces pelvic bone loss through ...,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3...,3630201
2,Stem Cell Health and Tissue Regeneration in Mi...,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1...,11988870
3,Microgravity Reduces the Differentiation and R...,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7...,7998608
4,Microgravity validation of a novel system for ...,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...,5587110


## Download xml files

### Subtask:
For each paper ID, download the full text XML using the provided API and save it to a temporary directory.


**Reasoning**:
Import necessary libraries, create a temporary directory, define the API URL, and iterate through the PMC IDs to download the XML files.



In [ ]:
import os
import requests
import time
import pandas as pd

# Temporary directory
temp_dir = 'temp_xml_downloads'
os.makedirs(temp_dir, exist_ok=True)

# Base URL
api_url_base = (
    'https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi'
    '?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:{pmc_id}&metadataPrefix=pmc'
)

# Retry function with backoff
def download_xml(pmc_id, retries=3, backoff_factor=2):
    url = api_url_base.format(pmc_id=pmc_id)
    attempt = 0
    wait_time = 2  # initial wait time in seconds

    while attempt < retries:
        try:
            response = requests.get(url, timeout=30)

            if response.status_code == 200:
                # Save XML file
                file_path = os.path.join(temp_dir, f'{pmc_id}.xml')
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                print(f"✅ Downloaded XML for PMC ID: {pmc_id}")
                return True

            elif response.status_code in [400, 429]:
                attempt += 1
                print(f"⚠️ Failed (status {response.status_code}) for PMC ID {pmc_id}. "
                      f"Retry {attempt}/{retries} after {wait_time}s...")
                time.sleep(wait_time)
                wait_time *= backoff_factor  # exponential backoff

            else:
                print(f"❌ Permanent failure for PMC ID {pmc_id}. Status code: {response.status_code}")
                return False

        except requests.exceptions.RequestException as e:
            attempt += 1
            print(f"⚠️ Error for PMC ID {pmc_id}: {e}. Retry {attempt}/{retries} after {wait_time}s...")
            time.sleep(wait_time)
            wait_time *= backoff_factor

    print(f"❌ Giving up on PMC ID {pmc_id} after {retries} retries.")
    return False


# Loop through dataframe
for pmc_id in df['PMC_ID']:
    if pd.notna(pmc_id):
        download_xml(str(int(pmc_id)))  # make sure it's string
        time.sleep(0.2)  # small delay between requests

NameError: name 'df' is not defined

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("temp_xml_downloads", 'zip', "temp_xml_downloads")
files.download("temp_xml_downloads.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!mv /content/temp_xml_downloads /content/drive/MyDrive/public

# Process xml and extract metadata (edit)

### Subtask:
Parse each XML file, extract the relevant metadata (title, authors, year, journal, abstract, keywords, mission), and save it as a JSON file. Also, save the original XML file.


**Reasoning**:
Iterate through the downloaded XML files, parse each file, extract the required metadata, structure it into a dictionary, and save it as a JSON file. Also, save the original XML file in the specified directory structure.



In [ ]:
import os
import json
import xml.etree.ElementTree as ET
import shutil

# Paths
temp_dir = '/content/drive/MyDrive/public/temp_xml_downloads'
output_base_dir = '/content/drive/MyDrive/public/data/papers'
master_json_path = '/content/drive/MyDrive/public/data/publications.json'

publications_data = {}
failed_files = []

os.makedirs(output_base_dir, exist_ok=True)

for filename in os.listdir(temp_dir):
    if filename.endswith('.xml'):
        pmc_id = os.path.splitext(filename)[0]
        xml_file_path = os.path.join(temp_dir, filename)

        paper_output_dir = os.path.join(output_base_dir, pmc_id)
        os.makedirs(paper_output_dir, exist_ok=True)

        output_json_path = os.path.join(paper_output_dir, f'{pmc_id}.json')
        output_xml_path = os.path.join(paper_output_dir, f'{pmc_id}.xml')

        try:
            tree = ET.parse(xml_file_path)
            root = tree.getroot()

            # --- THREE-STAGE CHECK TO FIND THE CORRECT DATA FORMAT ---
            ns = None
            prefix = ''
            article_meta = None

            # Stage 1: Try default namespace ('d:')
            try_ns = {'d': 'https://jats.nlm.nih.gov/ns/archiving/1.4/'}
            article_meta = root.find('.//d:article-meta', try_ns)
            if article_meta is not None:
                ns = try_ns
                prefix = 'd:'

            # Stage 2: If that fails, try prefixed namespace ('jats:')
            if article_meta is None:
                try_ns = {'jats': 'https://jats.nlm.nih.gov/ns/archiving/1.4/'}
                article_meta = root.find('.//jats:article-meta', try_ns)
                if article_meta is not None:
                    ns = try_ns
                    prefix = 'jats:'

            # Stage 3: If both fail, assume no namespace
            if article_meta is None:
                article_meta = root.find('.//article-meta')
                if article_meta is not None:
                    prefix = ''
                    ns = None # No namespace map needed

            if article_meta is None:
                raise ValueError("Could not find <article-meta> tag. XML format is unrecognized.")

            # --- Helper function for clean searching ---
            def find(element, path):
                return element.find(path, ns) if ns else element.find(path)

            def findall(element, path):
                return element.findall(path, ns) if ns else element.findall(path)

            # --- Extract metadata using the detected prefix ---
            title_element = find(article_meta, f'.//{prefix}article-title')
            title_text = ''.join(title_element.itertext()).strip() if title_element is not None else 'N/A'

            affiliations = []
            for aff in findall(article_meta, f'.//{prefix}aff'):
                full_affiliation_text = ''.join(aff.itertext()).strip()
                cleaned_text = ' '.join(full_affiliation_text.split())
                affiliations.append(cleaned_text)

            authors = []
            for author in findall(article_meta, f'.//{prefix}contrib-group/{prefix}contrib[@contrib-type="author"]'):
                given_name = find(author, f'.//{prefix}given-names')
                surname = find(author, f'.//{prefix}surname')

                full_name_parts = []
                if given_name is not None and given_name.text:
                    full_name_parts.append(given_name.text)
                if surname is not None and surname.text:
                    full_name_parts.append(surname.text)

                if full_name_parts:
                    author_name = " ".join(full_name_parts)
                    authors.append(author_name)

            year_element = find(article_meta, f'.//{prefix}pub-date[@pub-type="epub"]/{prefix}year')
            month_element = find(article_meta, f'.//{prefix}pub-date[@pub-type="epub"]/{prefix}month')
            day_element = find(article_meta, f'.//{prefix}pub-date[@pub-type="epub"]/{prefix}day')

            year = year_element.text if year_element is not None else 'YYYY'
            month = f"{int(month_element.text):02d}" if month_element is not None else 'MM'
            day = f"{int(day_element.text):02d}" if day_element is not None else 'DD'
            publication_date = f"{year}-{month}-{day}"

            journal_element = find(article_meta, f'.//{prefix}journal-title')
            journal_text = journal_element.text if journal_element is not None else 'N/A'

            volume_element = find(article_meta, f'.//{prefix}volume')
            volume_text = volume_element.text if volume_element is not None else 'N/A'

            issue_element = find(article_meta, f'.//{prefix}issue')
            issue_text = issue_element.text if issue_element is not None else 'N/A'

            category_element = find(root, f'.//{prefix}article-categories//{prefix}subject')
            category_text = category_element.text.strip() if category_element is not None else 'N/A'

            abstract_element = find(article_meta, f'.//{prefix}abstract')
            abstract_text = ''.join(abstract_element.itertext()).strip() if abstract_element is not None else 'N/A'

            keywords = [kw.text.strip() for kw in findall(article_meta, f'.//{prefix}kwd-group/{prefix}kwd') if kw.text]

            metadata = {
                'pmc_id': pmc_id,
                'title': title_text,
                'authors': authors,
                'affiliations': affiliations,
                'publication_date': publication_date,
                'year': publication_date.split('-')[0], # ✅ FIX: Add the 'year' field back
                'journal': journal_text,
                'volume': volume_text,
                'issue': issue_text,
                'category': category_text,
                'abstract': abstract_text,
                'keywords': keywords,
            }

            with open(output_json_path, 'w', encoding='utf-8') as f:
                json.dump(metadata, f, indent=4, ensure_ascii=False)

            shutil.copy(xml_file_path, output_xml_path)
            publications_data[pmc_id] = metadata

        except Exception as e:
            print(f"Failed to process {filename}: {e}")
            failed_files.append(filename)

with open(master_json_path, 'w', encoding='utf-8') as f:
    json.dump(publications_data, f, indent=4, ensure_ascii=False)

print(f"\nProcessing complete.")
print(f"Successfully processed: {len(publications_data)} files.")
print(f"Failed to process: {len(failed_files)} files.")
if failed_files:
    print("Failed files:", failed_files)


Processing complete.
Successfully processed: 494 files.
Failed to process: 0 files.


## Organize files

### Subtask:
Create the specified folder structure (`./public/data/papers/{pmc_id}/`) and move the metadata JSON and full text XML files into the corresponding directories.


## Build master file

### Subtask:
Create a `publications.json` file containing all the extracted metadata.


**Reasoning**:
Initialize an empty dictionary, iterate through the directories in `./public/data/papers/`, read the JSON file in each directory, add the data to the dictionary, and finally save the dictionary as `./public/data/publications.json`.



In [ ]:
import os
import json

publications_data = {}

# ✅ Save location inside Google Drive (My Drive folder)
output_base_dir = '/content/drive/MyDrive/public/data/papers'
master_json_path = '/content/drive/MyDrive/public/data/publications.json'

# Make sure directories exist
os.makedirs(output_base_dir, exist_ok=True)
os.makedirs(os.path.dirname(master_json_path), exist_ok=True)

# Collect all JSON metadata files from subdirectories
if os.path.exists(output_base_dir):
    for pmc_id_dir in os.listdir(output_base_dir):
        pmc_id_path = os.path.join(output_base_dir, pmc_id_dir)
        if os.path.isdir(pmc_id_path):
            json_file_path = os.path.join(pmc_id_path, f'{pmc_id_dir}.json')
            if os.path.exists(json_file_path):
                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        metadata = json.load(f)
                        publications_data[pmc_id_dir] = metadata
                except Exception as e:
                    print(f"Error reading JSON file {json_file_path}: {e}")

# ✅ Save the combined master JSON in Google Drive
with open(master_json_path, 'w', encoding='utf-8') as f:
    json.dump(publications_data, f, indent=4, ensure_ascii=False)

print(f"Created master JSON file at {master_json_path} with {len(publications_data)} entries.")

Created master JSON file at /content/drive/MyDrive/public/data/publications.json with 494 entries.


## Summary:

### Data Analysis Key Findings

*   The initial dataset contained 434 paper URLs with corresponding titles.
*   PMC IDs were successfully extracted from the URLs for all entries.
*   XML full texts were downloaded for the majority of papers, with some failures due to API issues (Bad Request and Too Many Requests).
*   Metadata (title, authors, year, journal, abstract, keywords, mission) was successfully extracted from 434 downloaded XML files.
*   The extracted metadata and original XML files were organized into a directory structure `./public/data/papers/{pmc_id}/`.
*   A master JSON file (`./public/data/publications.json`) was created containing the extracted metadata for all 434 processed papers.

### Insights or Next Steps

*   Investigate the failed XML downloads to understand the specific reasons for the errors (e.g., invalid PMC IDs, temporary API issues) and attempt to re-download or handle them appropriately.
*   Enhance the XML parsing to extract more structured information, such as affiliations for authors, specific sections of the abstract, or funding information, if needed for future analysis.


# KGraph
Process the paper URLs from the "SB_publication_PMC.csv" file, download the full text XML for each paper using the provided API, extract specific metadata (title, authors, year, journal, abstract, keywords, mission) from the XML, and save the extracted metadata as a JSON file and the original XML file for each paper in a structured directory format (`./public/data/papers/{pmc_id}/`) within Google Drive. Finally, create a master "publications.json" file in `./public/data/` within Google Drive containing all the extracted metadata.

## Organize files

### Subtask:
Mount Google Drive, create the specified folder structure (`./public/data/papers/{pmc_id}/`) within Google Drive, and move the metadata JSON and full text XML files into the corresponding directories.


**Reasoning**:
Mount Google Drive, create the necessary directory structure, and move the processed JSON and XML files from the temporary directory to the corresponding directories within Google Drive.



In [ ]:
import os
import json
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from google import genai
from google.genai import types
from getpass import getpass

# CONFIGURATION

# The root directory for the project on your Google Drive.
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/public/data'

# --- API KEY LOADING (secrets only — NEVER hardcode keys) ---
# Option 1 (Colab): create a Secret named GEMINI_API_KEYS containing one or
#   more keys joined by commas (e.g. "AIzaSy...,AIzaSy..."), and enable
#   notebook access for it.
# Option 2 (anywhere): export GEMINI_API_KEYS=... before running.
# Option 3 (fallback): you'll be prompted via getpass at runtime.
def _load_gemini_keys():
    raw = None
    try:
        from google.colab import userdata  # only available inside Colab
        raw = userdata.get('GEMINI_API_KEYS')
    except Exception:
        pass
    if not raw:
        raw = os.environ.get('GEMINI_API_KEYS')
    if not raw:
        raw = getpass('Paste your Gemini API key(s), comma-separated: ')
    keys = [k.strip() for k in raw.split(',') if k.strip()]
    if not keys:
        raise ValueError('At least one Gemini API key is required.')
    return keys

GEMINI_API_KEYS = _load_gemini_keys()

# Define derived paths and constants
ROOT_PAPERS_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'papers')
OUTPUT_GRAPH_FILE = os.path.join(DRIVE_PROJECT_ROOT, 'knowledge_graph.json')
EXTRACT_FILENAME = 'graph_extract.json'
XML_FILENAME = 'full_text.xml'


# API Key Rotator (round-robins across keys to spread quota)
class ApiKeyManager:
    def __init__(self, keys):
        if not keys:
            raise ValueError("API key list cannot be empty.")
        self.keys = keys
        self.current_index = 0

    def get_next_key(self):
        key = self.keys[self.current_index]
        self.current_index = (self.current_index + 1) % len(self.keys)
        return key

api_key_manager = ApiKeyManager(GEMINI_API_KEYS)

print("\n--- Setup Complete ---")
print(f"Project Root: {DRIVE_PROJECT_ROOT}")
print(f"Papers Directory: {ROOT_PAPERS_DIR}")
print(f"Loaded {len(GEMINI_API_KEYS)} Gemini key(s) from secrets.")


In [ ]:
import os
from tqdm.notebook import tqdm

def verify_and_standardize_papers():
    """
    Verifies the existence of paper folders and standardizes the XML filename.
    """
    print(f"Scanning for paper directories in: {ROOT_PAPERS_DIR}")

    if not os.path.isdir(ROOT_PAPERS_DIR):
        print(f"❌ ERROR: The directory does not exist: {ROOT_PAPERS_DIR}")
        print("Please check the `DRIVE_PROJECT_ROOT` path in the first setup cell.")
        return

    paper_dirs = [d for d in os.listdir(ROOT_PAPERS_DIR) if os.path.isdir(os.path.join(ROOT_PAPERS_DIR, d))]

    if not paper_dirs:
        print("❌ WARNING: No paper subdirectories found.")
        print("Please ensure your paper folders (e.g., '2824534') are inside the 'papers' directory.")
        return

    print(f"Found {len(paper_dirs)} potential paper directories. Now verifying and standardizing...")

    renamed_count = 0
    verified_count = 0

    for pmc_id in tqdm(paper_dirs, desc="Verifying Papers"):
        paper_dir = os.path.join(ROOT_PAPERS_DIR, pmc_id)

        # Define the source and destination paths for the XML file
        original_xml_path = os.path.join(paper_dir, f"{pmc_id}.xml")
        standard_xml_path = os.path.join(paper_dir, XML_FILENAME) # XML_FILENAME is 'full_text.xml' from the setup cell

        # If the standard file already exists, we're good
        if os.path.exists(standard_xml_path):
            verified_count += 1
            continue

        # If the original file exists, rename it
        if os.path.exists(original_xml_path):
            try:
                os.rename(original_xml_path, standard_xml_path)
                renamed_count += 1
                verified_count += 1
            except Exception as e:
                print(f"  [Error] Could not rename file for {pmc_id}: {e}")
        else:
            print(f"  [Warning] No XML file found for paper {pmc_id}.")

    print("\n--- Verification and Standardization Complete ---")
    print(f"✅ Verified {verified_count} paper folders with a standard '{XML_FILENAME}' file.")
    if renamed_count > 0:
      print(f"  > Renamed {renamed_count} files to the standard name.")
    print("Ready to proceed to Step 2: AI Extraction.")

verify_and_standardize_papers()

Scanning for paper directories in: /content/drive/MyDrive/public/data/papers
Found 494 potential paper directories. Now verifying and standardizing...


Verifying Papers:   0%|          | 0/494 [00:00<?, ?it/s]


--- Verification and Standardization Complete ---
✅ Verified 494 paper folders with a standard 'full_text.xml' file.
Ready to proceed to Step 2: AI Extraction.


In [ ]:
PROMPT_TEMPLATE = """
Analyze the provided scientific paper text. Your task is to extract entities (nodes), the relationships between them (edges), and key scientific claims.

**Task Requirements:**

1.  **Identify Nodes:**
    * For each entity, provide its `name` as it appears in the text.
    * Provide a `normalized_name`. This is crucial. For example, "mice" and "Mus musculus" should both be normalized to "Mus musculus". "Micro-gravity" should be normalized to "Microgravity".
    * Assign a `label` from the **Allowed Node Labels**.
    * If applicable, assign a `subtype` from the **Allowed Subtypes**.
    * Provide the `provenance` (the exact sentence) and a `confidence` score.

2.  **Identify Relationships (Edges):**
    * Identify relationships between the **normalized names** of the Nodes.
    * Assign a `relation_type` from the **Allowed Relationship Types**.
    * Provide the `evidence` sentence and a `confidence` score.

3.  **Extract Scientific Claims:**
    * Extract standalone scientific statements.
    * List the `involved_entity_normalized_names`.
    * Provide a `certainty` level ('asserted' or 'speculative').

**Allowed Labels and Subtypes (Use these exact values):**
* **Node Label:** `Organism`
    * **Subtypes:** `Mammals`, `Plants`, `Microbes/Pathogens`, `Human`, `Fish/Cephalopod/Other`, `Rat Organs`, `Mouse Organs`, `Fungi (Yeast and Microbial Community Components)`, `Animals (Vertebrates and Invertebrates)`, `Bacteria and Archaea (Microbiome and Pathogens)`
* **Node Label:** `Condition`
    * **Subtypes:** `Disease`, `Environmental`, `Spaceflight`, `Microgravity`, `Radiation`, `Stress`, `Aging`, `Nutritional`
* **Node Label:** `Molecule`
    * **Subtypes:** `Gene`, `Protein`, `RNA`, `Metabolite`, `Hormone`, `Chemical Compound`, `Signaling Molecule`
    * **Gene Subtypes (extended):** `Oncogene`, `Tumor Suppressor`, `Transcription Factor`, `Housekeeping Gene`, `Receptor Gene`, `Structural Gene`, `Mitochondrial Gene`
* **Node Label:** `Process`
    * **Subtypes:** `Biological Process`, `Molecular Process`, `Physiological Process`, `Pathological Process`, `Experimental Procedure`
* **Node Label:** `Anatomy`
    * **Subtypes:** `Organ`, `Tissue`, `Cellular Structure`, `Subcellular Structure`, `Skeletal Structure`, `Muscle`, `Nervous System`, `Circulatory System`
    * **Extended Subtypes:** `Respiratory System`, `Digestive System`, `Reproductive System`, `Immune System`, `Sensory Organ`, `Skin/Derma`, `Cartilage`, `Connective Tissue`
* **Node Label:** `Claim`
    * **Subtypes:** `Experimental Finding`, `Theoretical Assertion`, `Observation`, `Hypothesis`
    * **Extended Subtypes:** `Prediction`, `Correlation`, `Causal Inference`, `Speculative Mechanism`, `Negative Result`, `Validation/Replication`
* **Node Label:** `Technology`
    * **Subtypes:** `Spacecraft`, `Instrument`, `Sensor`, `Sequencing`, `Microscopy`, `Bioinformatics Tool`, `Computational Model`
    * **Extended Subtypes:** `Imaging Technology`, `Gene Editing`, `CRISPR`, `Omics Platform (Genomics/Proteomics/Metabolomics)`, `Wearable Device`, `Simulation Platform`, `Lab-on-a-Chip`, `AI/ML Model`, `Data Processing Pipeline`
* **Node Label:** `Tissue/Cell Type`
    * **Subtypes:** `Stem Cell`, `Immune Cell`, `Bone Cell`, `Muscle Cell`, `Neuron`, `Epithelial Cell`, `Endothelial Cell`
* **Node Label:** `Methodological Details`
    * **Subtypes:** `Experimental Design`, `Control`, `Sample Size`, `Statistical Method`, `Assay`, `Measurement`
* **Node Label:** `Data_Source`
    * **Subtypes:** `Database`, `Repository`, `Publication`, `Experiment`, `Simulation`
* **Node Label:** `Biological_Subject`
    * **Subtypes:** `Model Organism`, `Patient`, `Cohort`, `Sample`, `Cell Line`, `Population`
* **Node Label:** `Protein_Domain`
    * **Subtypes:** `Catalytic Domain`, `Binding Domain`, `Regulatory Domain`, `Structural Domain`

**Allowed Relationship Types:**
`ASSOCIATED_WITH`, `OCCURS_IN`, `REGULATES`, `MODULATES`, `AFFECTS_ANATOMY`, `MAKES_CLAIM`, `USES_TECHNIQUE`, `INVESTIGATES`, `HAS_PHENOTYPE`, `PARTICIPATES_IN`, `IS_A_MEMBER_OF`, `CAUSES`

**Output Rules:**
* Return **ONLY** a valid JSON document. No other text.
* The JSON **must** follow the exact schema provided below. Do not add extra keys.

**JSON Output Schema (Adhere to this structure exactly):**
{{
  "nodes": [
    {{
      "name": "string",
      "normalized_name": "string",
      "label": "string (from Allowed Labels)",
      "subtype": "string (from Allowed Subtypes)",
      "provenance": "string (the exact sentence)",
      "confidence": "float (0.0 to 1.0)"
    }}
  ],
  "edges": [
    {{
      "source": "string (a normalized_name from your nodes list)",
      "target": "string (a normalized_name from your nodes list)",
      "relation_type": "string (from Allowed Relationship Types)",
      "evidence": "string (the exact sentence)",
      "confidence": "float (0.0 to 1.0)"
    }}
  ],
  "claims": [
    {{
      "statement": "string (the scientific claim)",
      "involved_entity_normalized_names": ["string"],
      "certainty": "string ('asserted' or 'speculative')"
    }}
  ]
}}

---
**Input Paper Text:**
{paper_full_text}
"""



def run_gemini_extraction():
    """Finds all paper XMLs and runs Gemini extraction on them."""
    paper_dirs = [d for d in os.listdir(ROOT_PAPERS_DIR) if os.path.isdir(os.path.join(ROOT_PAPERS_DIR, d))]
    print(f"Found {len(paper_dirs)} paper directories to process.")

    # CHANGE 1: Slice the list to process only the first 5 papers.
    for pmc_id in tqdm(paper_dirs, desc="Extracting with Gemini"):
        paper_dir = os.path.join(ROOT_PAPERS_DIR, pmc_id)
        xml_path = os.path.join(paper_dir, XML_FILENAME)
        output_path = os.path.join(paper_dir, EXTRACT_FILENAME)

        if os.path.exists(output_path):
            # We print existing files too if we find them in the first 5
            try:
                with open(output_path, 'r', encoding='utf-8') as f:
                    print(f"\n--- Output for {pmc_id} (from existing file) ---")
                    print(f.read())
                    print("--------------------------------------------------\n")
                continue
            except Exception as e:
                print(f"  [Error] Could not read existing file for {pmc_id}: {e}")
                continue


        if not os.path.exists(xml_path):
            print(f"  [Warning] XML not found for {pmc_id}, skipping.")
            continue

        try:
            with open(xml_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f.read(), 'lxml-xml')
                article = soup.find('article')
                if not article:
                    print(f"  [Warning] Could not find <article> tag in {pmc_id}, skipping.")
                    continue
                paper_text = article.get_text(separator=' ', strip=True)
                paper_title = soup.find('article-title').get_text(strip=True) if soup.find('article-title') else "Title not found"

            final_prompt = PROMPT_TEMPLATE.format(paper_full_text=paper_text)

            # --- **FIXED CODE START (FOR OLDER LIBRARY VERSION)** ---

            # Get the next API key and initialize the client
            api_key = api_key_manager.get_next_key()
            client = genai.Client(api_key=api_key)

            model_name = "gemini-flash-lite-latest"

            # **THE FIX:** The parameter is 'config', and its value is a dictionary.
            # This corrects the original 'unexpected keyword argument' error.
            response = client.models.generate_content(
                model=model_name,
                contents=final_prompt,
                config={
                    "thinking_config": types.ThinkingConfig(thinking_budget=-1,),
                    "response_mime_type": "application/json"
                },
            )

            # The response text should be a valid JSON string
            extracted_data = json.loads(response.text)

            # Construct the final JSON object, adding the paper_info block
            result_json_dict = {
                "paper_info": {"title": paper_title, "pmc_id": pmc_id},
                **extracted_data
            }

            # --- **FIXED CODE END** ---

            # Save the JSON string to file
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(result_json_dict, f, indent=2)

            # CHANGE 2: Print the formatted JSON output to the console.
            # print(f"\n--- Output for {pmc_id} ---")
            # print(json.dumps(result_json_dict, indent=2))
            # print("---------------------------\n")


            # Respect API rate limits
            time.sleep(3)

        except Exception as e:
            print(f"  [Error] Failed processing {pmc_id}: {e}")

    print("\n✅ AI extraction process complete for the first 5 papers.")

# Run the extraction
run_gemini_extraction()

Found 494 paper directories to process.


Extracting with Gemini:   0%|          | 0/494 [00:00<?, ?it/s]


--- Output for 10764921 (from existing file) ---
{
  "paper_info": {
    "title": "Effects of altered gravity on growth and morphology inWolffia globosaimplications for bioregenerative life support systems and space-based agriculture",
    "pmc_id": "10764921"
  },
  "nodes": [
    {
      "name": "Wolffia globosa",
      "normalized_name": "Wolffia globosa",
      "label": "Organism",
      "subtype": "Plants",
      "provenance": "This study examines the impact of altered gravity conditions on the growth and morphological responses of Wolffia globosa (commonly known as \u201c water lentils \u201d or \u201c duckweed \u201d), assessing its potential as a space crop.",
      "confidence": 1.0
    },
    {
      "name": "water lentils",
      "normalized_name": "Wolffia globosa",
      "label": "Organism",
      "subtype": "Plants",
      "provenance": "Wolffia globosa (commonly known as \u201c water lentils \u201d or \u201c duckweed \u201d)",
      "confidence": 1.0
    },
    {
      

In [ ]:
import os
import json
import re
from tqdm.notebook import tqdm

def create_global_id(node_data):
    """Creates a deterministic, URL-friendly ID from a node's data."""
    label = node_data.get('label', 'unknown').lower()
    norm_name = node_data.get('normalized_name', '').lower()
    # Replace any non-alphanumeric characters with an underscore
    safe_name = re.sub(r'[^a-z0-9]+', '_', norm_name).strip('_')
    return f"{label}:{safe_name}"

def build_final_knowledge_graph():
    """
    Consolidates individual graph extracts into a single, globally consistent
    knowledge graph. It tracks which papers mention each node and builds a
    comprehensive edge list.
    """
    print("--- 🚀 Starting Knowledge Graph Consolidation ---")
    all_extracts_data = []

    # --- Phase 1: Load all individual graph extracts ---
    print(f"Phase 1: Loading all '{EXTRACT_FILENAME}' files...")
    for root, _, files in os.walk(ROOT_PAPERS_DIR):
        if EXTRACT_FILENAME in files:
            file_path = os.path.join(root, EXTRACT_FILENAME)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    all_extracts_data.append(data)
            except Exception as e:
                print(f"  [Warning] Could not load or parse {file_path}: {e}")

    if not all_extracts_data:
        print("\n--- ❌ ERROR: No valid extract files found. Aborting. ---")
        return

    print(f"  > Successfully loaded {len(all_extracts_data)} extract files.")

    # --- Phase 2: Aggregate all unique nodes and track mentions ---
    print("\nPhase 2: Aggregating unique nodes and tracking paper mentions...")
    master_nodes_map = {}
    for data in all_extracts_data:
        pmc_id = data.get('paper_info', {}).get('pmc_id', 'unknown')
        for node in data.get('nodes', []):
            norm_name = node.get('normalized_name')
            if not norm_name:
                continue

            if norm_name not in master_nodes_map:
                # First time seeing this node, store its full data
                master_nodes_map[norm_name] = node
                master_nodes_map[norm_name]['paper_mentions'] = [pmc_id] # Start tracking mentions
            else:
                # Node already exists, just add the new paper mention if not already there
                if pmc_id not in master_nodes_map[norm_name]['paper_mentions']:
                    master_nodes_map[norm_name]['paper_mentions'].append(pmc_id)

    print(f"  > Discovered {len(master_nodes_map)} unique concept nodes.")

    # --- Phase 3: Build the final nodes list (concepts + papers) ---
    print("\nPhase 3: Generating final node list with global IDs...")
    final_nodes_list = []
    node_name_to_id = {} # Map normalized_name to the new global ID

    # Process concept nodes
    for norm_name, node_data in master_nodes_map.items():
        node_id = create_global_id(node_data)
        node_data['id'] = node_id
        final_nodes_list.append(node_data)
        node_name_to_id[norm_name] = node_id

    # Process paper nodes
    for data in all_extracts_data:
        paper_info = data.get('paper_info', {})
        pmc_id = paper_info.get('pmc_id')
        if not pmc_id:
            continue

        paper_id = f"paper:{pmc_id}"
        paper_node_data = {
            'id': paper_id,
            'label': 'Paper',
            'name': paper_info.get('title', f"Paper {pmc_id}"),
            'normalized_name': paper_info.get('title', f"Paper {pmc_id}"),
            'pmc_id': pmc_id
        }
        final_nodes_list.append(paper_node_data)

    print(f"  > Generated a total of {len(final_nodes_list)} nodes (concepts + papers).")


    # --- Phase 4: Build the final, de-duplicated edge list ---
    print("\nPhase 4: Building final, de-duplicated edge list...")
    master_edges = []
    edge_set = set() # Use a set of tuples to efficiently track unique edges

    for data in tqdm(all_extracts_data, desc="Processing Edges"):
        paper_info = data.get('paper_info', {})
        pmc_id = paper_info.get('pmc_id')
        if not pmc_id:
            continue

        paper_id = f"paper:{pmc_id}"

        # Create edges from the Paper to the concepts it contains
        for node in data.get('nodes', []):
            target_id = node_name_to_id.get(node.get('normalized_name'))
            if target_id:
                edge_tuple = (paper_id, target_id, 'CONTAINS')
                if edge_tuple not in edge_set:
                    master_edges.append({
                        "source": paper_id,
                        "target": target_id,
                        "relation_type": "CONTAINS",
                        "evidence": f"Entity mentioned in paper PMC{pmc_id}"
                    })
                    edge_set.add(edge_tuple)

        # Create edges between concepts (the corrected part)
        for edge in data.get('edges', []):
            # ✨ THE FIX: Use `edge.get('source')` and `edge.get('target')`
            source_norm_name = edge.get('source')
            target_norm_name = edge.get('target')

            source_id = node_name_to_id.get(source_norm_name)
            target_id = node_name_to_id.get(target_norm_name)

            if source_id and target_id:
                relation_type = edge.get('relation_type', 'ASSOCIATED_WITH')
                edge_tuple = (source_id, target_id, relation_type)
                if edge_tuple not in edge_set:
                    master_edges.append({
                        "source": source_id,
                        "target": target_id,
                        "relation_type": relation_type,
                        "evidence": edge.get('evidence'),
                        "confidence": edge.get('confidence', 0.8) # Add confidence if available
                    })
                    edge_set.add(edge_tuple)

    print(f"  > Created {len(master_edges)} unique edges.")

    # --- Phase 5: Save the final knowledge graph ---
    print(f"\nPhase 5: Writing final graph to: {OUTPUT_GRAPH_FILE}")
    final_graph = {"nodes": final_nodes_list, "edges": master_edges}
    try:
        with open(OUTPUT_GRAPH_FILE, 'w', encoding='utf-8') as f:
            json.dump(final_graph, f, indent=2)
        print("\n✅ Knowledge Graph consolidation complete!")
        print(f"   Final graph contains {len(final_nodes_list)} nodes and {len(master_edges)} edges.")
    except Exception as e:
        print(f"\n--- ❌ ERROR: Could not write to output file: {e} ---")


# --- Run the Consolidation Process ---
build_final_knowledge_graph()

--- 🚀 Starting Knowledge Graph Consolidation ---
Phase 1: Loading all 'graph_extract.json' files...
  > Successfully loaded 459 extract files.

Phase 2: Aggregating unique nodes and tracking paper mentions...
  > Discovered 5891 unique concept nodes.

Phase 3: Generating final node list with global IDs...
  > Generated a total of 6350 nodes (concepts + papers).

Phase 4: Building final, de-duplicated edge list...


Processing Edges:   0%|          | 0/459 [00:00<?, ?it/s]

  > Created 13030 unique edges.

Phase 5: Writing final graph to: /content/drive/MyDrive/public/data/knowledge_graph.json

✅ Knowledge Graph consolidation complete!
   Final graph contains 6350 nodes and 13030 edges.


In [ ]:
!pip install pyvis

In [ ]:
import json
from pyvis.network import Network
from tqdm.notebook import tqdm
import pandas as pd

# Define the path to your knowledge graph file
GRAPH_JSON_PATH = '/content/drive/MyDrive/public/data/knowledge_graph.json'

print(f"Loading graph data from {GRAPH_JSON_PATH}...")
with open(GRAPH_JSON_PATH, 'r', encoding='utf-8') as f:
    graph_data = json.load(f)

nodes = graph_data.get('nodes', [])
edges = graph_data.get('edges', [])

print(f"Loaded {len(nodes)} nodes and {len(edges)} edges.")

# Create a Pyvis network object
net = Network(height='800px', width='100%', notebook=True, cdn_resources='in_line', heading='Star Engine Knowledge Graph')

# --- Add Nodes to the Graph ---
print("Adding nodes to the visualization...")
for node in tqdm(nodes, desc="Processing Nodes"):
    node_id = node.get('id')
    label = node.get('label', 'Unknown')
    display_name = node.get('name', node_id)

    # Create a hover title with more details
    title_text = f"ID: {node_id}\nLabel: {label}"
    if 'pmc_id' in node:
        title_text += f"\nPMC ID: {node['pmc_id']}"
    if 'paper_mentions' in node:
        title_text += f"\nMentioned in: {len(node['paper_mentions'])} papers"

    net.add_node(
        n_id=node_id,
        label=display_name,
        title=title_text,
        group=label
    )

# --- Add Edges to the Graph ---
print("Adding edges to the visualization...")
for edge in tqdm(edges, desc="Processing Edges"):
    source_id = edge.get('source')
    target_id = edge.get('target')
    relation = edge.get('relation_type', 'related')

    # Create a hover title for the edge
    title_text = f"Relation: {relation}"
    if 'evidence' in edge and edge['evidence'] is not None:
        evidence = str(edge['evidence']) # Ensure evidence is a string
        title_text += f"\nEvidence: {evidence[:200]}{'...' if len(evidence) > 200 else ''}"

    # ✨ THE FIX: Changed 'target' to 'to'
    net.add_edge(
        source=source_id,
        to=target_id,
        title=title_text
    )

# --- Configure Physics and UI ---
net.show_buttons(filter_=['physics'])

# Generate and display the graph
print("\nGenerating interactive graph... (This may take a moment)")
net.show('knowledge_graph.html')
print("✅ Done! Interactive graph is displayed below.")

Loading graph data from /content/drive/MyDrive/public/data/knowledge_graph.json...
Loaded 6350 nodes and 13030 edges.
Adding nodes to the visualization...


Processing Nodes:   0%|          | 0/6350 [00:00<?, ?it/s]

Adding edges to the visualization...


Processing Edges:   0%|          | 0/13030 [00:00<?, ?it/s]


Generating interactive graph... (This may take a moment)
knowledge_graph.html
✅ Done! Interactive graph is displayed below.
